# Planners-05c — Le différentiel d'atteignabilité : ce que l'ajout d'une primitive rend possible

**Navigation** : [Index](../../README.md) | [Jumeau Python](Planners-05c-Differentiel-Atteignabilite-Python.ipynb)

> « Les agents modifient leur vocabulaire » est une formule nébuleuse. Ce carnet la rend **testable**.

Le lake [`planning_lean`](../planning_lean/) — consommé ici, jamais modifié — prouve sans `sorry` que toute trajectoire réelle reste une trajectoire possible dans la **dynamique relaxée sans delete** (`run π s ⊆ runR π s`, `Planning/Relaxation.lean:56`, lemme clé `step ⊆ stepR` en `Planning/Strips.lean:58`), d'où l'admissibilité $h^+ \le h^*$ (`Planning/Admissibility.lean:41`). C'est un théorème sur **deux mondes issus du même vocabulaire** : la relaxation élargit ce que les deletes interdisaient.

La question de ce carnet est d'une autre nature :

> **quelle nouvelle classe de plans devient atteignable après l'ajout d'une primitive ?**

C'est un **différentiel d'atteignabilité** : $A_t \hookrightarrow A_{t+1}$, et l'on mesure ce que l'élargissement du vocabulaire rend possible que l'approfondissement de la recherche dans $A_t$ ne pouvait pas. Le protocole, en quatre pas, est implémenté respectivement par les cellules de code **7, 10, 15 et 17** :

1. **Un problème où aucune politique disponible n'atteint le but** — vérifié, pas supposé.
2. **L'agent cherche davantage dans le même $A_t$** — plus de budget, meilleure heuristique — et **échoue toujours**. Sans cette mesure de contrôle, on ne sait pas si l'extension a servi.
3. **L'agent est autorisé à inventer une primitive** — nommée, et **payante**.
4. **L'extension rend le but atteignable** — le delta est mesuré en buts, en coûts, en taille d'espace.

Ce que le protocole sépare : *chercher davantage dans un espace d'actions donné* contre *élargir cet espace*. C'est aussi le banc du **différentiel de Laborit** — l'écart entre approfondir et élargir. Le companion formel de la relaxation est [Planners-5b](Planners-05b-Lean-Relaxation-Lean.ipynb) ; le présent carnet en est le **consommateur expérimental**.

## Le moteur STRIPS, minimal et déterministe

Tout est en **std pur** (aucune dépendance externe) : le sujet est le protocole de mesure, pas un solveur. Un état est un `BTreeSet<String>` de propositions, un opérateur STRIPS un triplet (préconditions, ajouts, suppressions). Deux dynamiques cohabitent, exactement comme dans le lake :

- `successeurs` — la dynamique **réelle** (`step` : adds puis dels) ;
- `successeurs_relaxes` — la dynamique **sans delete** (`stepR` : adds seulement), la même construction que celle sur laquelle `step_subset_stepR` (`Planning/Strips.lean:58`) porte.

L’itération est triée partout : deux exécutions donnent les mêmes nombres (la mesure est déterministe, exigence de la série).

In [2]:
use std::collections::{BTreeSet, HashMap, HashSet, VecDeque};

/// Etat : ensemble de propositions triees (Hash + Eq, jamais imprime brut).
type Etat = BTreeSet<String>;

/// Operateur STRIPS : nom, preconditions, ajouts, suppressions (cout 1).
#[derive(Clone)]
struct Operateur {
    nom: String,
    precond: Etat,
    ajouts: Etat,
    dels: Etat,
}

impl Operateur {
    fn applicable(&self, etat: &Etat) -> bool {
        self.precond.is_subset(etat)
    }
}

/// Dynamique reelle (step) : tous les etats successeurs, ordre deterministe.
fn successeurs(etat: &Etat, ops: &[Operateur]) -> Vec<(String, Etat)> {
    let mut tri: Vec<&Operateur> = ops.iter().collect();
    tri.sort_by(|a, b| a.nom.cmp(&b.nom));
    let mut out: Vec<(String, Etat)> = Vec::new();
    for op in tri {
        if op.applicable(etat) {
            let mut t = etat.clone();
            for p in op.ajouts.iter() {
                t.insert(p.clone());
            }
            for p in op.dels.iter() {
                t.remove(p);
            }
            out.push((op.nom.clone(), t));
        }
    }
    out
}

/// Dynamique sans delete (stepR) : l'etat ne fait que croitre.
fn successeurs_relaxes(etat: &Etat, ops: &[Operateur]) -> Vec<(String, Etat)> {
    let mut tri: Vec<&Operateur> = ops.iter().collect();
    tri.sort_by(|a, b| a.nom.cmp(&b.nom));
    let mut out: Vec<(String, Etat)> = Vec::new();
    for op in tri {
        if op.applicable(etat) && !op.ajouts.is_subset(etat) {
            let mut t = etat.clone();
            for p in op.ajouts.iter() {
                t.insert(p.clone());
            }
            out.push((op.nom.clone(), t));
        }
    }
    out
}

/// Resultat du BFS : distances, peres, expansions et ordre de decouverte.
struct Bfs {
    dist: HashMap<Etat, i64>,
    peres: HashMap<Etat, Option<(Etat, String)>>,
    expansions: i64,
    ordre: Vec<Etat>,
}

/// Enumere EXACTEMENT l'ensemble atteignable (dynamique reelle).
fn bfs(initial: &Etat, ops: &[Operateur]) -> Bfs {
    let mut dist: HashMap<Etat, i64> = HashMap::new();
    let mut peres: HashMap<Etat, Option<(Etat, String)>> = HashMap::new();
    dist.insert(initial.clone(), 0);
    peres.insert(initial.clone(), None);
    let mut ordre: Vec<Etat> = vec![initial.clone()];
    let mut file: VecDeque<Etat> = VecDeque::new();
    file.push_back(initial.clone());
    let mut expansions: i64 = 0;
    while let Some(s) = file.pop_front() {
        expansions += 1;
        let g = *dist.get(&s).unwrap_or(&0);
        for (nom, t) in successeurs(&s, ops) {
            if !dist.contains_key(&t) {
                dist.insert(t.clone(), g + 1);
                peres.insert(t.clone(), Some((s.clone(), nom)));
                ordre.push(t.clone());
                file.push_back(t);
            }
        }
    }
    Bfs { dist, peres, expansions, ordre }
}

/// Remontee des peres puis inversion : le plan dans l'ordre d'execution.
fn reconstruire(peres: &HashMap<Etat, Option<(Etat, String)>>, but: &Etat) -> Vec<String> {
    let mut plan: Vec<String> = Vec::new();
    let mut s = but.clone();
    while let Some(Some((pere, nom))) = peres.get(&s) {
        plan.push(nom.clone());
        s = pere.clone();
    }
    plan.reverse();
    plan
}

/// BFS dans le monde sans delete : distance relaxee de chaque proposition
/// (premiere apparition). C'est la base de l'heuristique h_max.
fn distances_relaxees(initial: &Etat, ops: &[Operateur]) -> HashMap<String, i64> {
    let mut dist_atoms: HashMap<String, i64> = HashMap::new();
    for p in initial.iter() {
        dist_atoms.insert(p.clone(), 0);
    }
    let mut vu: HashSet<Etat> = HashSet::new();
    vu.insert(initial.clone());
    let mut file: VecDeque<(Etat, i64)> = VecDeque::new();
    file.push_back((initial.clone(), 0));
    while let Some((s, d)) = file.pop_front() {
        for (_, t) in successeurs_relaxes(&s, ops) {
            if vu.insert(t.clone()) {
                for p in t.difference(&s) {
                    dist_atoms.entry(p.clone()).or_insert(d + 1);
                }
                file.push_back((t, d + 1));
            }
        }
    }
    dist_atoms
}

/// h_max : max des distances relaxees des atomes du but. Admissible,
/// calculable exactement -- la version calculable de la chaine h_max <= h+ <= h*
/// garantiee par le lake (Admissibility.lean:41).
/// None = infini (float('inf') en Python).
fn h_max(etat: &Etat, but_conjonctif: &Etat, ops: &[Operateur]) -> Option<i64> {
    let dr = distances_relaxees(etat, ops);
    let mut best: Option<i64> = None;
    for b in but_conjonctif.iter() {
        match dr.get(b) {
            None => return None,
            Some(v) => {
                let vv = *v;
                best = Some(match best {
                    None => vv,
                    Some(cur) => cur.max(vv),
                });
            }
        }
    }
    best
}

println!("Moteur STRIPS charge : step (reel) + stepR (relaxe sans delete).");

Moteur STRIPS charge : step (reel) + stepR (relaxe sans delete).


### Les deux dynamiques, face à face avec le lake

La correspondance avec le lake est terme à terme : là où `Planning/Strips.lean` définit un `Action F` (préconditions, ajouts, suppressions) et deux transitions — `step` (la réelle : ajouts **puis** suppressions) et `stepR` (la relaxée : ajouts **seulement**) — le moteur ci-dessus définit la même paire. Le lemme `step_subset_stepR` (`Strips.lean:58`) dit pourquoi la relaxée ne peut qu’élargir : appliquer un opérateur sans faire les suppressions rend l’état obtenu **sur-ensemble** de l’état réel — un état réel est toujours un état relaxé possible, jamais l’inverse.

Deux choix de mesure méritent d’être dits :

- **Déterminisme** : les opérateurs sont itérés triés par nom, les états sont des `BTreeSet<String>` — deux exécutions produisent les mêmes nombres. Une mesure qui dérive d’une exécution à l’autre ne peut pas étalonner un différentiel.
- **Pourquoi `h_max` et pas `h⁺` direct** : le coût du plan relaxé **optimal** ($h^+$) est NP-difficile à calculer exactement ; $h_{max}$ — le maximum des distances relaxées par atome — se calcule exactement par BFS et minore $h^+$. La chaîne complète mesurable est donc $h_{max} \le h^+ \le h^*$ : le lake garantit le maillon droit (`Admissibility.lean:41`), le carnet mesure les deux extrémités calculables.

## Le domaine : deux rives, un gouffre, un radeau qui n’existe pas encore

Une grille $6 \times 3$ où la **colonne 3 est un gouffre** (aucune case) : rive gauche (colonnes 0-2), rive droite (colonnes 4-5). Le robot démarre en $(0,0)$. Sur la rive gauche : une **planchette** en $(1,1)$ et une **clé** en $(2,0)$. Sur la rive droite : un **trésor** en $(4,1)$.

Le vocabulaire disponible $A_t$ : **se déplacer** (4 directions, case cible existante) et **ramasser** (être sur la case de l’objet). Trois buts :

| But | Proposition | Rive |
|---|---|---|
| $g_1$ | atteindre la balise en $(5,2)$ | droite |
| $g_2$ | détenir le trésor | droite |
| $g_3$ | détenir la clé | **gauche** |

$g_3$ est le **témoin de santé** : s’il échoue, c’est le moteur qui est en cause, pas le gouffre.

In [3]:
use std::collections::BTreeSet;

/// Etat : ensemble de propositions triees.
type Etat = BTreeSet<String>;

/// Operateur STRIPS : nom, preconditions, ajouts, suppressions (cout 1).
#[derive(Clone)]
struct Operateur {
    nom: String,
    precond: Etat,
    ajouts: Etat,
    dels: Etat,
}

impl Operateur {
    fn applicable(&self, etat: &Etat) -> bool {
        self.precond.is_subset(etat)
    }
}

const COLS_GAUCHE: [i64; 3] = [0, 1, 2];
const COLS_DROITE: [i64; 2] = [4, 5];
const RANGS: [i64; 3] = [0, 1, 2];

fn prop(c: i64, r: i64) -> String {
    format!("at-x{c}-y{r}")
}

fn singleton(s: &str) -> Etat {
    let mut e: Etat = BTreeSet::new();
    e.insert(s.to_string());
    e
}

/// CASES : colonnes 0-2 et 4-5, rangs 0-2, triees lexicographiquement.
/// colonne 3 = gouffre
fn cases() -> Vec<(i64, i64)> {
    let mut v: Vec<(i64, i64)> = Vec::new();
    for c in COLS_GAUCHE.iter().chain(COLS_DROITE.iter()) {
        for r in RANGS.iter() {
            v.push((*c, *r));
        }
    }
    v.sort();
    v.dedup();
    v
}

fn initial() -> Etat {
    let mut s: Etat = BTreeSet::new();
    s.insert(prop(0, 0));
    s.insert("plank-at-x1-y1".to_string());
    s.insert("key-at-x2-y0".to_string());
    s.insert("treasure-at-x4-y1".to_string());
    s
}

/// Operateurs de base : deplacements (case cible existante) + ramassages.
/// L'ordre de construction est sans effet : tout parcours passe par le tri par nom.
fn ops_de_base() -> Vec<Operateur> {
    let cs = cases();
    let mut ops: Vec<Operateur> = Vec::new();
    for (c, r) in cs.iter() {
        for (dc, dr, nom_dir) in [(1, 0, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud")].iter() {
            let cible = (*c + *dc, *r + *dr);
            if cs.contains(&cible) {
                ops.push(Operateur {
                    nom: format!("move-{nom_dir}-x{c}-y{r}"),
                    precond: singleton(&prop(*c, *r)),
                    ajouts: singleton(&prop(cible.0, cible.1)),
                    dels: singleton(&prop(*c, *r)),
                });
            }
        }
    }
    for (objet, cell) in [("plank", (1, 1)), ("key", (2, 0)), ("treasure", (4, 1))].iter() {
        let (c, r) = *cell;
        let mut pre: Etat = BTreeSet::new();
        pre.insert(prop(c, r));
        pre.insert(format!("{objet}-at-x{c}-y{r}"));
        let mut aj: Etat = BTreeSet::new();
        aj.insert(format!("holding-{objet}"));
        let mut dl: Etat = BTreeSet::new();
        dl.insert(format!("{objet}-at-x{c}-y{r}"));
        ops.push(Operateur { nom: format!("pickup-{objet}"), precond: pre, ajouts: aj, dels: dl });
    }
    ops
}

/// Liste Python (crochets, apostrophes, ', ') -- le {:?} de Rust met des guillemets doubles.
fn py_list(items: &[&str]) -> String {
    let mut s = String::from("[");
    for (i, it) in items.iter().enumerate() {
        if i > 0 {
            s.push_str(", ");
        }
        s.push('\'');
        s.push_str(it);
        s.push('\'');
    }
    s.push(']');
    s
}

/// Les trois buts, dans l'ordre d'insertion (g1, g2, g3) -- qui est aussi l'ordre trie.
fn buts() -> Vec<(&'static str, Etat)> {
    vec![
        ("g1: balise (5,2)", singleton(&prop(5, 2))),
        ("g2: detenir le tresor", singleton("holding-treasure")),
        ("g3: detenir la cle", singleton("holding-key")),
    ]
}

let a_t = ops_de_base();
let b = buts();
let noms: Vec<&str> = b.iter().map(|(n, _)| *n).collect();
let n_moves = a_t.iter().filter(|o| o.nom.starts_with("move")).count();
let n_pickups = a_t.iter().filter(|o| o.nom.starts_with("pickup")).count();
println!("Cases existantes : {} (colonne 3 = gouffre)", cases().len());
println!("Operateurs A_t   : {} ({} moves + {} pickups)", a_t.len(), n_moves, n_pickups);
println!("Buts             : {}", py_list(&noms));

Cases existantes : 15 (colonne 3 = gouffre)


Operateurs A_t   : 41 (38 moves + 3 pickups)


Buts             : ['g1: balise (5,2)', 'g2: detenir le tresor', 'g3: detenir la cle']


## Pas 1 — Vérifier l’inatteignabilité, pas la supposer

L’énumération exhaustive de l’ensemble atteignable est un **certificat fini** : si l’espace est énuméré en entier et que le but n’y figure pas, l’inatteignabilité est **exacte** — pas un délai dépassé, pas une impression. C’est l’exigence du protocole : le pas 1 ne dit pas « on n’a pas trouvé », il dit « **il n’y a pas** ».

In [4]:
use std::collections::{BTreeSet, HashMap, HashSet, VecDeque};

// Un état est un ensemble de propositions ; un opérateur porte ses effets STRIPS.
type Etat = BTreeSet<String>;
#[derive(Clone)]
struct Operateur { nom: String, precond: Etat, ajouts: Etat, dels: Etat }
impl Operateur {
    fn applicable(&self, etat: &Etat) -> bool { self.precond.is_subset(etat) }
}
// La dynamique réelle applique les ajouts puis les suppressions, par nom trié.
fn successeurs(etat: &Etat, ops: &[Operateur]) -> Vec<(String, Etat)> {
    let mut tri: Vec<&Operateur> = ops.iter().collect();
    tri.sort_by(|a, b| a.nom.cmp(&b.nom));
    let mut out = Vec::new();
    for op in tri {
        if op.applicable(etat) {
            let mut t = etat.clone();
            t.extend(op.ajouts.iter().cloned());
            for p in &op.dels { t.remove(p); }
            out.push((op.nom.clone(), t));
        }
    }
    out
}
// La dynamique relaxée ignore les suppressions et les actions sans effet nouveau.
fn successeurs_relaxes(etat: &Etat, ops: &[Operateur]) -> Vec<(String, Etat)> {
    let mut tri: Vec<&Operateur> = ops.iter().collect();
    tri.sort_by(|a, b| a.nom.cmp(&b.nom));
    let mut out = Vec::new();
    for op in tri {
        if op.applicable(etat) && !op.ajouts.is_subset(etat) {
            let mut t = etat.clone();
            t.extend(op.ajouts.iter().cloned());
            out.push((op.nom.clone(), t));
        }
    }
    out
}
// Le BFS conserve aussi les pères et l'ordre de découverte.
struct Bfs {
    dist: HashMap<Etat, i64>,
    peres: HashMap<Etat, Option<(Etat, String)>>,
    expansions: i64,
    ordre: Vec<Etat>,
}
fn bfs(initial: &Etat, ops: &[Operateur]) -> Bfs {
    let mut dist = HashMap::new();
    let mut peres = HashMap::new();
    let mut ordre = vec![initial.clone()];
    let mut file = VecDeque::new();
    dist.insert(initial.clone(), 0);
    peres.insert(initial.clone(), None);
    file.push_back(initial.clone());
    let mut expansions = 0;
    while let Some(s) = file.pop_front() {
        expansions += 1;
        let g = dist[&s];
        for (nom, t) in successeurs(&s, ops) {
            if !dist.contains_key(&t) {
                dist.insert(t.clone(), g + 1);
                peres.insert(t.clone(), Some((s.clone(), nom)));
                ordre.push(t.clone());
                file.push_back(t);
            }
        }
    }
    Bfs { dist, peres, expansions, ordre }
}
// Remonter les pères puis inverser les actions reconstruit un plan.
fn reconstruire(peres: &HashMap<Etat, Option<(Etat, String)>>, but: &Etat) -> Vec<String> {
    let mut plan = Vec::new();
    let mut s = but.clone();
    while let Some(Some((pere, nom))) = peres.get(&s) {
        plan.push(nom.clone());
        s = pere.clone();
    }
    plan.reverse();
    plan
}
// Dans le BFS relaxé, la première apparition fixe la distance de chaque atome.
fn distances_relaxees(initial: &Etat, ops: &[Operateur]) -> HashMap<String, i64> {
    let mut distances = HashMap::new();
    for p in initial { distances.insert(p.clone(), 0); }
    let mut vu = HashSet::new();
    vu.insert(initial.clone());
    let mut file = VecDeque::new();
    file.push_back((initial.clone(), 0));
    while let Some((s, d)) = file.pop_front() {
        for (_, t) in successeurs_relaxes(&s, ops) {
            if vu.insert(t.clone()) {
                for p in t.difference(&s) { distances.entry(p.clone()).or_insert(d + 1); }
                file.push_back((t, d + 1));
            }
        }
    }
    distances
}
// h_max prend le maximum des distances atomiques ; None représente l'infini.
fn h_max(etat: &Etat, but: &Etat, ops: &[Operateur]) -> Option<i64> {
    let distances = distances_relaxees(etat, ops);
    let mut best: Option<i64> = None;
    for b in but {
        let v = *distances.get(b)?;
        best = Some(match best { None => v, Some(cur) => cur.max(v) });
    }
    best
}
// Construire la grille sans la colonne du gouffre et l'état initial.
const COLS_GAUCHE: [i64; 3] = [0, 1, 2];
const COLS_DROITE: [i64; 2] = [4, 5];
const RANGS: [i64; 3] = [0, 1, 2];
fn prop(c: i64, r: i64) -> String { format!("at-x{c}-y{r}") }
fn singleton(s: &str) -> Etat { let mut e = BTreeSet::new(); e.insert(s.to_string()); e }
fn cases() -> Vec<(i64, i64)> {
    let mut v = Vec::new();
    for c in COLS_GAUCHE.iter().chain(COLS_DROITE.iter()) {
        for r in RANGS.iter() { v.push((*c, *r)); }
    }
    v.sort(); v.dedup(); v
}
fn initial() -> Etat {
    let mut s = BTreeSet::new();
    s.insert(prop(0, 0)); s.insert("plank-at-x1-y1".to_string());
    s.insert("key-at-x2-y0".to_string()); s.insert("treasure-at-x4-y1".to_string()); s
}
// Générer les déplacements entre cases existantes et les ramassages.
fn ops_de_base() -> Vec<Operateur> {
    let cs = cases(); let mut ops = Vec::new();
    for (c, r) in cs.iter() {
        for (dc, dr, direction) in [(1, 0, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud")] {
            let cible = (*c + dc, *r + dr);
            if cs.contains(&cible) {
                ops.push(Operateur { nom: format!("move-{direction}-x{c}-y{r}"), precond: singleton(&prop(*c, *r)), ajouts: singleton(&prop(cible.0, cible.1)), dels: singleton(&prop(*c, *r)) });
            }
        }
    }
    for (objet, c, r) in [("plank", 1, 1), ("key", 2, 0), ("treasure", 4, 1)] {
        let mut pre = BTreeSet::new(); pre.insert(prop(c, r)); pre.insert(format!("{objet}-at-x{c}-y{r}"));
        let mut aj = BTreeSet::new(); aj.insert(format!("holding-{objet}"));
        let mut dl = BTreeSet::new(); dl.insert(format!("{objet}-at-x{c}-y{r}"));
        ops.push(Operateur { nom: format!("pickup-{objet}"), precond: pre, ajouts: aj, dels: dl });
    }
    ops
}
fn buts() -> Vec<(&'static str, Etat)> {
    vec![("g1: balise (5,2)", singleton(&prop(5, 2))), ("g2: detenir le tresor", singleton("holding-treasure")), ("g3: detenir la cle", singleton("holding-key"))]
}
// Formater les booléens et les listes comme Python.
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
fn py_list(items: &[&str]) -> String {
    let mut s = String::from("[");
    for (i, item) in items.iter().enumerate() {
        if i > 0 { s.push_str(", "); }
        s.push('\''); s.push_str(item); s.push('\'');
    }
    s.push(']'); s
}
// Énumérer exhaustivement les états et mesurer chaque but.
let etat_initial = initial();
let a_t = ops_de_base();
let liste_buts = buts();
let resultat = bfs(&etat_initial, &a_t);
println!("Ensemble atteignable dans A_t : {} etats, {} expansions (BFS exhaustif)", resultat.ordre.len(), resultat.expansions);
println!();
for (nom, but) in liste_buts.iter() {
    let ok = resultat.ordre.iter().any(|s| but.is_subset(s));
    let hstar = resultat.ordre.iter().filter_map(|s| if but.is_subset(s) { resultat.dist.get(s) } else { None }).min();
    let h_txt = match hstar { None => "infini".to_string(), Some(v) => v.to_string() };
    println!("{:28} atteignable={}   h*={}", nom, py_bool(ok), h_txt);
}
// Chercher toute proposition de la rive droite dans les états atteints.
let mut prop_droite: Vec<String> = Vec::new();
for s in resultat.ordre.iter() {
    for p in s.iter() {
        if p.starts_with("at-x4") || p.starts_with("at-x5") || p.starts_with("holding-treasure") { prop_droite.push(p.clone()); }
    }
}
prop_droite.sort(); prop_droite.dedup();
let liste = if prop_droite.is_empty() { "AUCUNE".to_string() } else { py_list(&prop_droite.iter().map(|s| s.as_str()).collect::<Vec<&str>>()) };
println!();
println!("Propositions de la rive droite presentes dans l'ensemble atteignable : {}", liste);

Ensemble atteignable dans A_t : 36 etats, 36 expansions (BFS exhaustif)


g1: balise (5,2)             atteignable=False   h*=infini


g2: detenir le tresor        atteignable=False   h*=infini


g3: detenir la cle           atteignable=True   h*=3


Propositions de la rive droite presentes dans l'ensemble atteignable : AUCUNE


### Lecture du pas 1 : le certificat et son architecture

L'ensemble atteignable vaut exactement **36 états** — et ce nombre n'est pas arbitraire : 9 cases de la rive gauche × 4 combinaisons d'inventaire (∅, planchette, clé, les deux). La rive droite apporte **zéro** état : aucune proposition `at-x4-*`/`at-x5-*`/`holding-treasure` n'apparaît jamais. Le BFS a énuméré l'espace **en entier** (36 expansions pour 36 états : chaque état a été dépilé exactement une fois) — l'inatteignabilité de $g_1$ et $g_2$ est donc un **certificat fini**, à distinguer soigneusement d'un échec de recherche : un solveur qui « n'a pas trouvé en 10 secondes » ne prouve rien ; une énumération complète qui ne contient pas le but prouve l'impossibilité.

Le témoin de santé fait son office : $g_3$ est atteignable avec $h^* = 3$ (deux déplacements jusqu'à la clé en $(2,0)$, un pickup). Le moteur marche, le gouffre seul est en cause.

## Pas 2 — La mesure de contrôle : chercher *davantage* dans le même $A_t$

C'est le pas que l'on a le plus envie de sauter, et le grain entier s'effondre sans lui. Trois niveaux d'effort, **même vocabulaire** :

1. **BFS exhaustif** — l'exploration complète (déjà faite, budget : 36 états) ;
2. **A\*** avec l'heuristique admissible $h_{max}$ — la meilleure heuristique calculable issue de la relaxation du lake (chaîne $h_{max} \le h^+ \le h^*$) ;
3. **IDDFS** (approfondissement itératif, profondeur 24) — l'effort systématique sans mémoire.

Si le but devient atteignable à un seul de ces niveaux, l'extension du pas 3 n'aurait rien mesuré : c'était un problème de recherche, pas de vocabulaire.

In [5]:
use std::cmp::Reverse;
use std::collections::{BTreeSet, BinaryHeap, HashMap, HashSet, VecDeque};

type Etat = BTreeSet<String>;
#[derive(Clone)]
struct Operateur { nom: String, precond: Etat, ajouts: Etat, dels: Etat }
impl Operateur {
    fn applicable(&self, etat: &Etat) -> bool { self.precond.is_subset(etat) }
}
fn successeurs(etat: &Etat, ops: &[Operateur]) -> Vec<(String, Etat)> {
    let mut tri: Vec<&Operateur> = ops.iter().collect();
    tri.sort_by(|a, b| a.nom.cmp(&b.nom));
    let mut out = Vec::new();
    for op in tri {
        if op.applicable(etat) {
            let mut t = etat.clone();
            t.extend(op.ajouts.iter().cloned());
            for p in &op.dels { t.remove(p); }
            out.push((op.nom.clone(), t));
        }
    }
    out
}
fn successeurs_relaxes(etat: &Etat, ops: &[Operateur]) -> Vec<(String, Etat)> {
    let mut tri: Vec<&Operateur> = ops.iter().collect();
    tri.sort_by(|a, b| a.nom.cmp(&b.nom));
    let mut out = Vec::new();
    for op in tri {
        if op.applicable(etat) && !op.ajouts.is_subset(etat) {
            let mut t = etat.clone();
            t.extend(op.ajouts.iter().cloned());
            out.push((op.nom.clone(), t));
        }
    }
    out
}
struct Bfs {
    dist: HashMap<Etat, i64>,
    peres: HashMap<Etat, Option<(Etat, String)>>,
    expansions: i64,
    ordre: Vec<Etat>,
}
fn bfs(initial: &Etat, ops: &[Operateur]) -> Bfs {
    let mut dist = HashMap::new();
    let mut peres = HashMap::new();
    let mut ordre = vec![initial.clone()];
    let mut file = VecDeque::new();
    dist.insert(initial.clone(), 0);
    peres.insert(initial.clone(), None);
    file.push_back(initial.clone());
    let mut expansions = 0;
    while let Some(s) = file.pop_front() {
        expansions += 1;
        let g = dist[&s];
        for (nom, t) in successeurs(&s, ops) {
            if !dist.contains_key(&t) {
                dist.insert(t.clone(), g + 1);
                peres.insert(t.clone(), Some((s.clone(), nom)));
                ordre.push(t.clone());
                file.push_back(t);
            }
        }
    }
    Bfs { dist, peres, expansions, ordre }
}
fn reconstruire(peres: &HashMap<Etat, Option<(Etat, String)>>, but: &Etat) -> Vec<String> {
    let mut plan = Vec::new();
    let mut s = but.clone();
    while let Some(Some((pere, nom))) = peres.get(&s) {
        plan.push(nom.clone());
        s = pere.clone();
    }
    plan.reverse();
    plan
}
fn distances_relaxees(initial: &Etat, ops: &[Operateur]) -> HashMap<String, i64> {
    let mut distances = HashMap::new();
    for p in initial { distances.insert(p.clone(), 0); }
    let mut vu = HashSet::new();
    vu.insert(initial.clone());
    let mut file = VecDeque::new();
    file.push_back((initial.clone(), 0));
    while let Some((s, d)) = file.pop_front() {
        for (_, t) in successeurs_relaxes(&s, ops) {
            if vu.insert(t.clone()) {
                for p in t.difference(&s) { distances.entry(p.clone()).or_insert(d + 1); }
                file.push_back((t, d + 1));
            }
        }
    }
    distances
}
fn h_max(etat: &Etat, but: &Etat, ops: &[Operateur]) -> Option<i64> {
    let dr = distances_relaxees(etat, ops);
    let mut best: Option<i64> = None;
    for b in but {
        let v = *dr.get(b)?;
        best = Some(match best { None => v, Some(cur) => cur.max(v) });
    }
    best
}

// Le compteur départage les états sans jamais comparer leur contenu.
fn astar_hmax(initial: &Etat, but: &Etat, ops: &[Operateur]) -> (bool, i64, Option<i64>) {
    let mut compteur = 0usize;
    let mut frontiere: BinaryHeap<Reverse<(i64, i64, usize)>> = BinaryHeap::new();
    let mut etats = vec![initial.clone()];
    let mut vu = HashSet::new();
    vu.insert(initial.clone());
    frontiere.push(Reverse((h_max(initial, but, ops).unwrap_or(i64::MAX), 0, compteur)));
    let mut expansions = 0;
    while let Some(Reverse((_f, g, c))) = frontiere.pop() {
        let s = etats[c].clone();
        expansions += 1;
        if but.is_subset(&s) { return (true, expansions, Some(g)); }
        for (_, t) in successeurs(&s, ops) {
            if vu.insert(t.clone()) {
                compteur += 1;
                etats.push(t.clone());
                let f = match h_max(&t, but, ops) { Some(h) => g + 1 + h, None => i64::MAX };
                frontiere.push(Reverse((f, g + 1, compteur)));
            }
        }
    }
    (false, expansions, None)
}
// La pile LIFO explore les successeurs dans l'ordre inverse de leur insertion.
fn iddfs(initial: &Etat, but: &Etat, ops: &[Operateur], prof_max: usize) -> (bool, i64, Option<usize>) {
    let mut expansions = 0;
    for limite in 0..=prof_max {
        let mut pile = vec![(initial.clone(), 0usize)];
        let mut chemin_vu = HashSet::new();
        while let Some((s, d)) = pile.pop() {
            expansions += 1;
            if but.is_subset(&s) { return (true, expansions, Some(d)); }
            if d < limite {
                for (_, t) in successeurs(&s, ops) {
                    if chemin_vu.insert((t.clone(), d + 1)) { pile.push((t, d + 1)); }
                }
            }
        }
    }
    (false, expansions, None)
}

const COLS_GAUCHE: [i64; 3] = [0, 1, 2];
const COLS_DROITE: [i64; 2] = [4, 5];
const RANGS: [i64; 3] = [0, 1, 2];
fn prop(c: i64, r: i64) -> String { format!("at-x{c}-y{r}") }
fn singleton(s: &str) -> Etat { let mut e = BTreeSet::new(); e.insert(s.to_string()); e }
fn cases() -> Vec<(i64, i64)> {
    let mut v = Vec::new();
    for c in COLS_GAUCHE.iter().chain(COLS_DROITE.iter()) {
        for r in RANGS.iter() { v.push((*c, *r)); }
    }
    v.sort(); v.dedup(); v
}
fn initial() -> Etat {
    let mut s = BTreeSet::new();
    s.insert(prop(0, 0)); s.insert("plank-at-x1-y1".to_string());
    s.insert("key-at-x2-y0".to_string()); s.insert("treasure-at-x4-y1".to_string()); s
}
fn ops_de_base() -> Vec<Operateur> {
    let cs = cases(); let mut ops = Vec::new();
    for (c, r) in cs.iter() {
        for (dc, dr, direction) in [(1, 0, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud")] {
            let cible = (*c + dc, *r + dr);
            if cs.contains(&cible) {
                ops.push(Operateur { nom: format!("move-{direction}-x{c}-y{r}"), precond: singleton(&prop(*c, *r)), ajouts: singleton(&prop(cible.0, cible.1)), dels: singleton(&prop(*c, *r)) });
            }
        }
    }
    for (objet, c, r) in [("plank", 1, 1), ("key", 2, 0), ("treasure", 4, 1)] {
        let mut pre = BTreeSet::new(); pre.insert(prop(c, r)); pre.insert(format!("{objet}-at-x{c}-y{r}"));
        let mut aj = BTreeSet::new(); aj.insert(format!("holding-{objet}"));
        let mut dl = BTreeSet::new(); dl.insert(format!("{objet}-at-x{c}-y{r}"));
        ops.push(Operateur { nom: format!("pickup-{objet}"), precond: pre, ajouts: aj, dels: dl });
    }
    ops
}
fn buts() -> Vec<(&'static str, Etat)> {
    vec![("g1: balise (5,2)", singleton(&prop(5, 2))), ("g2: detenir le tresor", singleton("holding-treasure")), ("g3: detenir la cle", singleton("holding-key"))]
}

let etat_initial = initial();
let a_t = ops_de_base();
let liste_buts = buts();
let resultat_bfs = bfs(&etat_initial, &a_t);
let mut rapport = Vec::new();
for (nom, but) in liste_buts.iter() {
    let ok_bfs = resultat_bfs.ordre.iter().any(|s| but.is_subset(s));
    let (ok_astar, exp_astar, _cout_astar) = astar_hmax(&etat_initial, but, &a_t);
    let (ok_iddfs, exp_iddfs, _profondeur_iddfs) = iddfs(&etat_initial, but, &a_t, 24);
    rapport.push((nom.to_string(), ok_bfs, resultat_bfs.expansions, ok_astar, exp_astar, ok_iddfs, exp_iddfs));
}
println!("{:28} {:>10} {:>18} {:>18}", "but", "BFS", "A*+h_max", "IDDFS(24)");
for (nom, b, eb, a, ea, i, ei) in rapport.iter() {
    let okko = if *b { "OK" } else { "ECHEC" };
    let okko2 = if *a { "OK" } else { "ECHEC" };
    let okko3 = if *i { "OK" } else { "ECHEC" };
    println!("{:28} {:>7}({:>3}) {:>8}({:>4}) {:>8}({:>5})", nom, okko, eb, okko2, ea, okko3, ei);
}
println!();
println!("Verdict du controle : g1 et g2 echouent aux TROIS niveaux d'effort.");
println!("L'espace atteignable est le MEME ({} etats) quel que soit le budget.", resultat_bfs.ordre.len());

but                                 BFS           A*+h_max          IDDFS(24)


g1: balise (5,2)               ECHEC( 36)    ECHEC(  36)    ECHEC( 4151)


g2: detenir le tresor          ECHEC( 36)    ECHEC(  36)    ECHEC( 4151)


g3: detenir la cle                OK( 36)       OK(   4)       OK(   24)


Verdict du controle : g1 et g2 echouent aux TROIS niveaux d'effort.


L'espace atteignable est le MEME (36 etats) quel que soit le budget.


### Lecture du pas 2 : l'asymétrie des budgets est la signature

Le tableau se lit en deux colonnes morales. Sur le but **atteignable** $g_3$, l'effort paie de façon spectaculaire : BFS énumère les 36 états, A\* avec $h_{max}$ n'en développe que **4** (l'heuristique admissible le mène droit au but), IDDFS en re-développe 24. C'est le régime ordinaire de la planification : *plus d'effort, une meilleure heuristique — le même problème devient tractable*.

Sur les buts **inatteignables**, l'effet disparaît : BFS et A\* rendent le même verdict en énumérant **le même espace de 36 états** — la meilleure heuristique du monde ne peut pas indiquer un chemin qui n'existe pas, elle ne peut qu'épuiser. IDDFS, lui, **brûle 4 151 expansions** — plus de cent fois la taille de l'espace, parce qu'il ré-explorre sans mémoire à chaque seuil de profondeur — pour conclure la même chose. C'est l'asymétrie qui authentifie le diagnostic :

- budgets qui **s'améliorent** sur un but → le problème était de *recherche* ;
- budgets qui **brûlent sans rien rapporter** → le problème est de *vocabulaire*.

Et le résultat négatif de la relaxation (section suivante) complète le verdict : même le monde relaxé — pourtant plus grand par construction — ne contient pas la rive droite. Le trou n'est pas dans les deletes, il est dans l'alphabet des actions.

## Ce que le lake garantit — et pourquoi la relaxation ne peut pas sauver $g_1$

Le théorème d'admissibilité (`Planning/Admissibility.lean:41`) dit : tout plan réel est un plan relaxé, donc $h^+ \le h^*$. La relaxation **sans delete** (`run ⊆ runR`, `Planning/Relaxation.lean:56`) est un élargissement de monde : elle rend atteignable tout ce que les suppressions d'effets interdisaient. Mesurons-la ici :

- sur la rive **gauche**, la relaxation minorer honnêtement : $h_{max} \le h^*$, parfois **strictement** — le but conjonctif « planchette **et** clé **et** revenir au départ » coûte 8 en réel mais paraît plus court au monde relaxé (les positions s'accumulent, les détours disparaissent) ;
- sur $g_1$/$g_2$, la distance relaxée est **infinie elle aussi** : aucun opérateur de $A_t$, même relaxé, ne produit une proposition de la rive droite. La relaxation élargit les mondes **par les deletes** ; une **primitive manquante** est un trou que même le monde relaxé ne comble pas.

In [6]:
use std::collections::{BTreeSet, HashMap, HashSet, VecDeque};

// Un état est un ensemble de propositions et un opérateur porte ses effets STRIPS.
type Etat = BTreeSet<String>;
#[derive(Clone)]
struct Operateur { nom: String, precond: Etat, ajouts: Etat, dels: Etat }
impl Operateur { fn applicable(&self, etat: &Etat) -> bool { self.precond.is_subset(etat) } }
// La dynamique réelle applique les ajouts puis les suppressions en ordre déterministe.
fn successeurs(etat: &Etat, ops: &[Operateur]) -> Vec<(String, Etat)> {
    let mut tri: Vec<&Operateur> = ops.iter().collect();
    tri.sort_by(|a, b| a.nom.cmp(&b.nom));
    let mut out = Vec::new();
    for op in tri {
        if op.applicable(etat) {
            let mut t = etat.clone(); t.extend(op.ajouts.iter().cloned());
            for p in &op.dels { t.remove(p); }
            out.push((op.nom.clone(), t));
        }
    }
    out
}
// La dynamique relaxée ne supprime rien et omet les actions sans effet nouveau.
fn successeurs_relaxes(etat: &Etat, ops: &[Operateur]) -> Vec<(String, Etat)> {
    let mut tri: Vec<&Operateur> = ops.iter().collect();
    tri.sort_by(|a, b| a.nom.cmp(&b.nom));
    let mut out = Vec::new();
    for op in tri {
        if op.applicable(etat) && !op.ajouts.is_subset(etat) {
            let mut t = etat.clone(); t.extend(op.ajouts.iter().cloned());
            out.push((op.nom.clone(), t));
        }
    }
    out
}
// Le BFS énumère les états et mémorise distances, pères et ordre de découverte.
struct Bfs {
    dist: HashMap<Etat, i64>,
    peres: HashMap<Etat, Option<(Etat, String)>>,
    expansions: i64,
    ordre: Vec<Etat>,
}
fn bfs(initial: &Etat, ops: &[Operateur]) -> Bfs {
    let mut dist = HashMap::new(); let mut peres = HashMap::new();
    let mut ordre = vec![initial.clone()]; let mut file = VecDeque::new();
    dist.insert(initial.clone(), 0); peres.insert(initial.clone(), None); file.push_back(initial.clone());
    let mut expansions = 0;
    while let Some(s) = file.pop_front() {
        expansions += 1;
        let g = dist[&s];
        for (nom, t) in successeurs(&s, ops) {
            if !dist.contains_key(&t) {
                dist.insert(t.clone(), g + 1); peres.insert(t.clone(), Some((s.clone(), nom)));
                ordre.push(t.clone()); file.push_back(t);
            }
        }
    }
    Bfs { dist, peres, expansions, ordre }
}
// Remonter les pères permet de reconstruire un plan dans l'ordre d'exécution.
fn reconstruire(peres: &HashMap<Etat, Option<(Etat, String)>>, but: &Etat) -> Vec<String> {
    let mut plan = Vec::new(); let mut s = but.clone();
    while let Some(Some((pere, nom))) = peres.get(&s) { plan.push(nom.clone()); s = pere.clone(); }
    plan.reverse(); plan
}
// Un BFS relaxé fixe la distance d'un atome à sa première apparition.
fn distances_relaxees(initial: &Etat, ops: &[Operateur]) -> HashMap<String, i64> {
    let mut distances = HashMap::new();
    for p in initial { distances.insert(p.clone(), 0); }
    let mut vu = HashSet::new(); vu.insert(initial.clone());
    let mut file = VecDeque::new(); file.push_back((initial.clone(), 0));
    while let Some((s, d)) = file.pop_front() {
        for (_, t) in successeurs_relaxes(&s, ops) {
            if vu.insert(t.clone()) {
                for p in t.difference(&s) { distances.entry(p.clone()).or_insert(d + 1); }
                file.push_back((t, d + 1));
            }
        }
    }
    distances
}
// h_max est le maximum des distances atomiques ; None représente l'infini.
fn h_max(etat: &Etat, but: &Etat, ops: &[Operateur]) -> Option<i64> {
    let dr = distances_relaxees(etat, ops);
    let mut best: Option<i64> = None;
    for b in but {
        let v = *dr.get(b)?;
        best = Some(match best { None => v, Some(cur) => cur.max(v) });
    }
    best
}
// Construire la grille sans la colonne du gouffre et l'état initial.
const COLS_GAUCHE: [i64; 3] = [0, 1, 2];
const COLS_DROITE: [i64; 2] = [4, 5];
const RANGS: [i64; 3] = [0, 1, 2];
fn prop(c: i64, r: i64) -> String { format!("at-x{c}-y{r}") }
fn singleton(s: &str) -> Etat { let mut e = BTreeSet::new(); e.insert(s.to_string()); e }
fn cases() -> Vec<(i64, i64)> {
    let mut v = Vec::new();
    for c in COLS_GAUCHE.iter().chain(COLS_DROITE.iter()) { for r in RANGS.iter() { v.push((*c, *r)); } }
    v.sort(); v.dedup(); v
}
fn initial() -> Etat {
    let mut s = BTreeSet::new();
    s.insert(prop(0, 0)); s.insert("plank-at-x1-y1".to_string());
    s.insert("key-at-x2-y0".to_string()); s.insert("treasure-at-x4-y1".to_string()); s
}
// Générer les déplacements et les ramassages du vocabulaire initial.
fn ops_de_base() -> Vec<Operateur> {
    let cs = cases(); let mut ops = Vec::new();
    for (c, r) in cs.iter() {
        for (dc, dr, direction) in [(1, 0, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud")] {
            let cible = (*c + dc, *r + dr);
            if cs.contains(&cible) {
                ops.push(Operateur { nom: format!("move-{direction}-x{c}-y{r}"), precond: singleton(&prop(*c, *r)), ajouts: singleton(&prop(cible.0, cible.1)), dels: singleton(&prop(*c, *r)) });
            }
        }
    }
    for (objet, c, r) in [("plank", 1, 1), ("key", 2, 0), ("treasure", 4, 1)] {
        let mut pre = BTreeSet::new(); pre.insert(prop(c, r)); pre.insert(format!("{objet}-at-x{c}-y{r}"));
        let mut aj = BTreeSet::new(); aj.insert(format!("holding-{objet}"));
        let mut dl = BTreeSet::new(); dl.insert(format!("{objet}-at-x{c}-y{r}"));
        ops.push(Operateur { nom: format!("pickup-{objet}"), precond: pre, ajouts: aj, dels: dl });
    }
    ops
}
fn buts() -> Vec<(&'static str, Etat)> {
    vec![("g1: balise (5,2)", singleton(&prop(5, 2))), ("g2: detenir le tresor", singleton("holding-treasure")), ("g3: detenir la cle", singleton("holding-key"))]
}
fn opt_txt(v: Option<i64>) -> String { match v { None => "inf".to_string(), Some(x) => x.to_string() } }
// (a) La relaxation minore, parfois strictement, sur la rive gauche.
let etat_initial = initial();
let a_t = ops_de_base();
let liste_buts = buts();
let resultat = bfs(&etat_initial, &a_t);
let mut but_conjonctif: Etat = BTreeSet::new();
but_conjonctif.insert("holding-plank".to_string());
but_conjonctif.insert("holding-key".to_string());
but_conjonctif.insert(prop(0, 0));
let hstar_conj = resultat.ordre.iter().filter_map(|s| if but_conjonctif.is_subset(s) { resultat.dist.get(s) } else { None }).min().copied();
let hmax_conj = h_max(&etat_initial, &but_conjonctif, &a_t);
let strict = match (hmax_conj, hstar_conj) { (Some(h), Some(s)) => h < s, _ => false };
println!("But conjonctif (planchette ET cle ET revenir au depart)");
println!("  h* (reel, BFS uniforme)  = {}", opt_txt(hstar_conj));
println!("  h_max (relaxe, exact)    = {}", opt_txt(hmax_conj));
println!("  chaine verifiee          : h_max ({}) <= h* ({})  -> {}", opt_txt(hmax_conj), opt_txt(hstar_conj), if strict { "STRICT" } else { "egalite" });
println!("  (le lake garantit h+ <= h* en Admissibility.lean:41 ; h_max <= h+ complete la chaine)");
// (b) Sur g1 et g2, tester aussi l'atteignabilité dans le monde relaxé.
for (nom, but) in liste_buts.iter().take(2) {
    let txt = match h_max(&etat_initial, but, &a_t) { None => "INFINI (relaxement inatteignable)".to_string(), Some(v) => v.to_string() };
    println!();
    println!("{}: h_max dans A_t = {}", nom, txt);
};

But conjonctif (planchette ET cle ET revenir au depart)


  h* (reel, BFS uniforme)  = 8


  h_max (relaxe, exact)    = 3


  chaine verifiee          : h_max (3) <= h* (8)  -> STRICT


  (le lake garantit h+ <= h* en Admissibility.lean:41 ; h_max <= h+ complete la chaine)


g1: balise (5,2): h_max dans A_t = INFINI (relaxement inatteignable)


g2: detenir le tresor: h_max dans A_t = INFINI (relaxement inatteignable)


## Pas 3 — La primitive nommée, et ce qu'elle coûte

Une extension **gratuite** ne mesure rien : si la nouvelle action ne coûte rien, le delta d'atteignabilité n'a pas de contrefactuel. La primitive inventée ici est le **radeau** :

> `raft-across-y{r}` — préconditions : être en $(2,r)$ **et** détenir la planchette. Effets : atterrir en $(4,r)$ ; **la planchette est consommée** (suppression `holding-plank`).

Son coût est double et visible dans les plans du pas 4 : (1) la planchette est **dépensée** — un seul trajet, pas d'aller-retour gratuit ; (2) l'aller chercher en $(1,1)$ est un **détour** imposé avant la traversée. Le vocabulaire s'élargit : $A_t \hookrightarrow A_{t+1}$.

In [7]:
use std::collections::BTreeSet;

// Etat et operateur STRIPS du domaine.
type Etat = BTreeSet<String>;

#[derive(Clone)]
struct Operateur {
    nom: String,
    precond: Etat,
    ajouts: Etat,
    dels: Etat,
}

impl Operateur {
    fn applicable(&self, etat: &Etat) -> bool {
        self.precond.is_subset(etat)
    }
}

// Grille 6 x 3 ; la colonne 3 demeure le gouffre.
const COLS_GAUCHE: [i64; 3] = [0, 1, 2];
const COLS_DROITE: [i64; 2] = [4, 5];
const RANGS: [i64; 3] = [0, 1, 2];
fn prop(c: i64, r: i64) -> String { format!("at-x{c}-y{r}") }
fn singleton(s: &str) -> Etat { let mut e = BTreeSet::new(); e.insert(s.to_string()); e }
fn cases() -> Vec<(i64, i64)> {
    let mut v = Vec::new();
    for c in COLS_GAUCHE.iter().chain(COLS_DROITE.iter()) { for r in RANGS.iter() { v.push((*c, *r)); } }
    v.sort(); v.dedup(); v
}

// A_t contient les deplacements et les ramassages disponibles.
fn ops_de_base() -> Vec<Operateur> {
    let cs = cases(); let mut ops = Vec::new();
    for (c, r) in cs.iter() {
        for (dc, dr, direction) in [(1, 0, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud")] {
            let cible = (*c + dc, *r + dr);
            if cs.contains(&cible) {
                ops.push(Operateur { nom: format!("move-{direction}-x{c}-y{r}"), precond: singleton(&prop(*c, *r)), ajouts: singleton(&prop(cible.0, cible.1)), dels: singleton(&prop(*c, *r)) });
            }
        }
    }
    for (objet, c, r) in [("plank", 1, 1), ("key", 2, 0), ("treasure", 4, 1)] {
        let mut pre = BTreeSet::new(); pre.insert(prop(c, r)); pre.insert(format!("{objet}-at-x{c}-y{r}"));
        let mut aj = BTreeSet::new(); aj.insert(format!("holding-{objet}"));
        let mut dl = BTreeSet::new(); dl.insert(format!("{objet}-at-x{c}-y{r}"));
        ops.push(Operateur { nom: format!("pickup-{objet}"), precond: pre, ajouts: aj, dels: dl });
    }
    ops
}

// A_t+1 ajoute trois radeaux ; chacun consomme la planchette.
fn ops_avec_radeau() -> Vec<Operateur> {
    let mut ops = ops_de_base();
    for r in RANGS.iter() {
        let mut pre = BTreeSet::new(); pre.insert(prop(2, *r)); pre.insert("holding-plank".to_string());
        let mut aj = BTreeSet::new(); aj.insert(prop(4, *r));
        let mut dl = BTreeSet::new(); dl.insert(prop(2, *r)); dl.insert("holding-plank".to_string());
        ops.push(Operateur { nom: format!("raft-across-y{r}"), precond: pre, ajouts: aj, dels: dl });
    }
    ops
}
fn py_list(items: &[String]) -> String {
    let mut s = String::from("[");
    for (i, item) in items.iter().enumerate() {
        if i > 0 { s.push_str(", "); }
        s.push('\''); s.push_str(item); s.push('\'');
    }
    s.push(']'); s
}

// Les tailles et les noms sont recalcules, jamais hardcodes.
let a_t = ops_de_base();
let a_t1 = ops_avec_radeau();
let mut nouveaux: Vec<(String, Vec<String>, Vec<String>, Vec<String>)> = Vec::new();
for op in a_t1.iter().filter(|op| op.nom.starts_with("raft-")) {
    nouveaux.push((op.nom.clone(), op.precond.iter().cloned().collect(), op.ajouts.iter().cloned().collect(), op.dels.iter().cloned().collect()));
}
let mut noms: Vec<String> = nouveaux.iter().map(|t| t.0.clone()).collect();
noms.sort();
println!("A_t   : {} operateurs", a_t.len());
println!("A_t+1 : {} operateurs (primitive ajoutee : {})", a_t1.len(), py_list(&noms));
for (nom, pre, aj, dl) in nouveaux.iter() {
    println!("  {}: precond={} ajouts={} supprime={}", nom, py_list(pre), py_list(aj), py_list(dl));
};

A_t   : 41 operateurs


A_t+1 : 44 operateurs (primitive ajoutee : ['raft-across-y0', 'raft-across-y1', 'raft-across-y2'])


  raft-across-y0: precond=['at-x2-y0', 'holding-plank'] ajouts=['at-x4-y0'] supprime=['at-x2-y0', 'holding-plank']


  raft-across-y1: precond=['at-x2-y1', 'holding-plank'] ajouts=['at-x4-y1'] supprime=['at-x2-y1', 'holding-plank']


  raft-across-y2: precond=['at-x2-y2', 'holding-plank'] ajouts=['at-x4-y2'] supprime=['at-x2-y2', 'holding-plank']


## Pas 4 — Le delta, mesuré

Trois mesures, pas une impression :

1. **En buts** : quels éléments de $G$ deviennent atteignables — le différentiel $\Delta = \text{atteignable}(A_{t+1}) \setminus \text{atteignable}(A_t)$, but par but ;
2. **En plans** : le plan pour $g_1$ et celui pour $g_2$, exhibés action par action avec leur coût $h^*$ ;
3. **En espace** : la taille de l'ensemble atteignable — le monde lui-même a grandi, ce n'est pas la recherche qui a approfondi.

In [8]:
use std::collections::{BTreeSet, HashMap, HashSet, VecDeque};

type Etat = BTreeSet<String>;
#[derive(Clone)]
struct Operateur { nom: String, precond: Etat, ajouts: Etat, dels: Etat }
impl Operateur { fn applicable(&self, etat: &Etat) -> bool { self.precond.is_subset(etat) } }
fn successeurs(etat: &Etat, ops: &[Operateur]) -> Vec<(String, Etat)> {
    let mut tri: Vec<&Operateur> = ops.iter().collect();
    tri.sort_by(|a, b| a.nom.cmp(&b.nom));
    let mut out = Vec::new();
    for op in tri {
        if op.applicable(etat) {
            let mut t = etat.clone(); t.extend(op.ajouts.iter().cloned());
            for p in &op.dels { t.remove(p); }
            out.push((op.nom.clone(), t));
        }
    }
    out
}
fn successeurs_relaxes(etat: &Etat, ops: &[Operateur]) -> Vec<(String, Etat)> {
    let mut tri: Vec<&Operateur> = ops.iter().collect();
    tri.sort_by(|a, b| a.nom.cmp(&b.nom));
    let mut out = Vec::new();
    for op in tri {
        if op.applicable(etat) && !op.ajouts.is_subset(etat) {
            let mut t = etat.clone(); t.extend(op.ajouts.iter().cloned());
            out.push((op.nom.clone(), t));
        }
    }
    out
}
struct Bfs {
    dist: HashMap<Etat, i64>,
    peres: HashMap<Etat, Option<(Etat, String)>>,
    expansions: i64,
    ordre: Vec<Etat>,
}
fn bfs(initial: &Etat, ops: &[Operateur]) -> Bfs {
    let mut dist = HashMap::new(); let mut peres = HashMap::new();
    let mut ordre = vec![initial.clone()]; let mut file = VecDeque::new();
    dist.insert(initial.clone(), 0); peres.insert(initial.clone(), None); file.push_back(initial.clone());
    let mut expansions = 0;
    while let Some(s) = file.pop_front() {
        expansions += 1;
        let g = dist[&s];
        for (nom, t) in successeurs(&s, ops) {
            if !dist.contains_key(&t) {
                dist.insert(t.clone(), g + 1); peres.insert(t.clone(), Some((s.clone(), nom)));
                ordre.push(t.clone()); file.push_back(t);
            }
        }
    }
    Bfs { dist, peres, expansions, ordre }
}
fn reconstruire(peres: &HashMap<Etat, Option<(Etat, String)>>, but: &Etat) -> Vec<String> {
    let mut plan = Vec::new(); let mut s = but.clone();
    while let Some(Some((pere, nom))) = peres.get(&s) { plan.push(nom.clone()); s = pere.clone(); }
    plan.reverse(); plan
}
fn distances_relaxees(initial: &Etat, ops: &[Operateur]) -> HashMap<String, i64> {
    let mut distances = HashMap::new();
    for p in initial { distances.insert(p.clone(), 0); }
    let mut vu = HashSet::new(); vu.insert(initial.clone());
    let mut file = VecDeque::new(); file.push_back((initial.clone(), 0));
    while let Some((s, d)) = file.pop_front() {
        for (_, t) in successeurs_relaxes(&s, ops) {
            if vu.insert(t.clone()) {
                for p in t.difference(&s) { distances.entry(p.clone()).or_insert(d + 1); }
                file.push_back((t, d + 1));
            }
        }
    }
    distances
}
fn h_max(etat: &Etat, but: &Etat, ops: &[Operateur]) -> Option<i64> {
    let dr = distances_relaxees(etat, ops);
    let mut best: Option<i64> = None;
    for b in but {
        let v = *dr.get(b)?;
        best = Some(match best { None => v, Some(cur) => cur.max(v) });
    }
    best
}
const COLS_GAUCHE: [i64; 3] = [0, 1, 2];
const COLS_DROITE: [i64; 2] = [4, 5];
const RANGS: [i64; 3] = [0, 1, 2];
fn prop(c: i64, r: i64) -> String { format!("at-x{c}-y{r}") }
fn singleton(s: &str) -> Etat { let mut e = BTreeSet::new(); e.insert(s.to_string()); e }
fn cases() -> Vec<(i64, i64)> {
    let mut v = Vec::new();
    for c in COLS_GAUCHE.iter().chain(COLS_DROITE.iter()) { for r in RANGS.iter() { v.push((*c, *r)); } }
    v.sort(); v.dedup(); v
}
fn initial() -> Etat {
    let mut s = BTreeSet::new();
    s.insert(prop(0, 0)); s.insert("plank-at-x1-y1".to_string());
    s.insert("key-at-x2-y0".to_string()); s.insert("treasure-at-x4-y1".to_string()); s
}
fn ops_de_base() -> Vec<Operateur> {
    let cs = cases(); let mut ops = Vec::new();
    for (c, r) in cs.iter() {
        for (dc, dr, direction) in [(1, 0, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud")] {
            let cible = (*c + dc, *r + dr);
            if cs.contains(&cible) {
                ops.push(Operateur { nom: format!("move-{direction}-x{c}-y{r}"), precond: singleton(&prop(*c, *r)), ajouts: singleton(&prop(cible.0, cible.1)), dels: singleton(&prop(*c, *r)) });
            }
        }
    }
    for (objet, c, r) in [("plank", 1, 1), ("key", 2, 0), ("treasure", 4, 1)] {
        let mut pre = BTreeSet::new(); pre.insert(prop(c, r)); pre.insert(format!("{objet}-at-x{c}-y{r}"));
        let mut aj = BTreeSet::new(); aj.insert(format!("holding-{objet}"));
        let mut dl = BTreeSet::new(); dl.insert(format!("{objet}-at-x{c}-y{r}"));
        ops.push(Operateur { nom: format!("pickup-{objet}"), precond: pre, ajouts: aj, dels: dl });
    }
    ops
}
fn ops_avec_radeau() -> Vec<Operateur> {
    let mut ops = ops_de_base();
    for r in RANGS.iter() {
        let mut pre = BTreeSet::new(); pre.insert(prop(2, *r)); pre.insert("holding-plank".to_string());
        let mut aj = BTreeSet::new(); aj.insert(prop(4, *r));
        let mut dl = BTreeSet::new(); dl.insert(prop(2, *r)); dl.insert("holding-plank".to_string());
        ops.push(Operateur { nom: format!("raft-across-y{r}"), precond: pre, ajouts: aj, dels: dl });
    }
    ops
}
fn buts() -> Vec<(&'static str, Etat)> {
    vec![("g1: balise (5,2)", singleton(&prop(5, 2))), ("g2: detenir le tresor", singleton("holding-treasure")), ("g3: detenir la cle", singleton("holding-key"))]
}
// Premier minimum dans l'ordre de decouverte du BFS.
fn distance_minimale(ordre: &[Etat], dist: &HashMap<Etat, i64>, but: &Etat) -> Option<(usize, i64)> {
    let mut meilleur: Option<(usize, i64)> = None;
    for (i, s) in ordre.iter().enumerate() {
        if but.is_subset(s) {
            let d = dist[s];
            match meilleur { None => meilleur = Some((i, d)), Some((_, cur)) if d < cur => meilleur = Some((i, d)), _ => {} }
        }
    }
    meilleur
}
let etat_initial = initial();
let a_t = ops_de_base();
let a_t1 = ops_avec_radeau();
let liste_buts = buts();
let avant = bfs(&etat_initial, &a_t);
let apres = bfs(&etat_initial, &a_t1);
println!("Ensemble atteignable : |A_t| = {} etats  ->  |A_t+1| = {} etats (+{})", avant.ordre.len(), apres.ordre.len(), apres.ordre.len() as i64 - avant.ordre.len() as i64);
println!();
println!("{:28} {:>8} {:>8} {:>10}  delta", "but", "A_t", "A_t+1", "h*(A_t+1)");
for (nom, but) in liste_buts.iter() {
    let ok_avant = avant.ordre.iter().any(|s| but.is_subset(s));
    let ok_apres = apres.ordre.iter().any(|s| but.is_subset(s));
    let h1 = match distance_minimale(&apres.ordre, &apres.dist, but) { None => format!("{:>10}", "inf"), Some((_, d)) => format!("{:>10}", d) };
    let delta = if ok_apres && !ok_avant { "NOUVELLEMENT ATTEIGNABLE" } else { "" };
    println!("{:28} {:>8} {:>8} {}  {}", nom, if ok_avant { "OK" } else { "ECHEC" }, if ok_apres { "OK" } else { "ECHEC" }, h1, delta);
}
let (i_g1, d_g1) = distance_minimale(&apres.ordre, &apres.dist, &liste_buts[0].1).unwrap_or((0, 0));
let (i_g2, d_g2) = distance_minimale(&apres.ordre, &apres.dist, &liste_buts[1].1).unwrap_or((0, 0));
println!();
println!("Plan g1 (balise, h* = {}) :", d_g1);
for (i, action) in reconstruire(&apres.peres, &apres.ordre[i_g1]).iter().enumerate() {
    println!("  {:2}. {}", i + 1, action);
}
println!();
println!("Plan g2 (tresor, h* = {}) :", d_g2);
for (i, action) in reconstruire(&apres.peres, &apres.ordre[i_g2]).iter().enumerate() {
    println!("  {:2}. {}", i + 1, action);
};

Ensemble atteignable : |A_t| = 36 etats  ->  |A_t+1| = 60 etats (+24)


but                               A_t    A_t+1  h*(A_t+1)  delta


g1: balise (5,2)                ECHEC       OK          7  NOUVELLEMENT ATTEIGNABLE


g2: detenir le tresor           ECHEC       OK          6  NOUVELLEMENT ATTEIGNABLE


g3: detenir la cle                 OK       OK          3  


Plan g1 (balise, h* = 7) :


... (10 outputs left out)

   3. pickup-plank


   4. move-est-x1-y1


   5. raft-across-y1


   6. pickup-treasure


### Lecture du delta : ce que les trois mesures disent ensemble

- **$\Delta = \{g_1, g_2\}$** — les deux buts de la rive droite deviennent atteignables ; $g_3$, le témoin de santé, était déjà atteignable et le reste : l'extension n'a rien cassé.
- **Les plans racontent le coût de la primitive** : le plan de $g_1$ commence par le détour $(0,0) \to (1,0) \to (1,1)$ pour la planchette, et la traversée **consomme** la planchette — si le but exigeait un retour, il faudrait réinventer. Le plan de $g_2$ suit le même squelette et s'arrête une case plus tôt : la traversée en **rangée 1** atterrit pile sur le trésor $(4,1)$ — un pickup de plus, deux déplacements de moins, d'où $h^*(g_2) = 6 < h^*(g_1) = 7$.
- **L'espace a grandi, pas la recherche** : l'ensemble atteignable passe de **36 à 60 états (+24)** — des états de la rive droite **existent maintenant**, alors que les trois niveaux d'effort du pas 2 énuméraient inlassablement les mêmes 36. C'est la différence entre approfondir $A_t$ et élargir en $A_{t+1}$ : le différentiel de Laborit, mesuré.

Et la relaxation ? Dans $A_{t+1}$, $h_{max}(g_1)$ devient fini — le monde relaxé suit le vocabulaire, jamais l'inverse.

In [9]:
use std::collections::{BTreeSet, HashMap, HashSet, VecDeque};

// Etat et operateur STRIPS du domaine.
type Etat = BTreeSet<String>;

#[derive(Clone)]
struct Operateur {
    nom: String,
    precond: Etat,
    ajouts: Etat,
    dels: Etat,
}

impl Operateur {
    fn applicable(&self, etat: &Etat) -> bool {
        self.precond.is_subset(etat)
    }
}

// Dynamique reelle, parcourue dans l'ordre lexical des noms.
fn successeurs(etat: &Etat, ops: &[Operateur]) -> Vec<(String, Etat)> {
    let mut tri: Vec<&Operateur> = ops.iter().collect();
    tri.sort_by(|a, b| a.nom.cmp(&b.nom));
    let mut out = Vec::new();
    for op in tri {
        if op.applicable(etat) {
            let mut t = etat.clone();
            t.extend(op.ajouts.iter().cloned());
            for p in op.dels.iter() { t.remove(p); }
            out.push((op.nom.clone(), t));
        }
    }
    out
}

// Dynamique relaxee sans delete, utilisee pour le calcul de h_max.
fn successeurs_relaxes(etat: &Etat, ops: &[Operateur]) -> Vec<(String, Etat)> {
    let mut tri: Vec<&Operateur> = ops.iter().collect();
    tri.sort_by(|a, b| a.nom.cmp(&b.nom));
    let mut out = Vec::new();
    for op in tri {
        if op.applicable(etat) && !op.ajouts.is_subset(etat) {
            let mut t = etat.clone();
            t.extend(op.ajouts.iter().cloned());
            out.push((op.nom.clone(), t));
        }
    }
    out
}

// Distance relaxee de premiere apparition pour chaque proposition.
fn distances_relaxees(initial: &Etat, ops: &[Operateur]) -> HashMap<String, i64> {
    let mut dist_atoms: HashMap<String, i64> = HashMap::new();
    for p in initial.iter() { dist_atoms.insert(p.clone(), 0); }
    let mut vu: HashSet<Etat> = HashSet::new(); vu.insert(initial.clone());
    let mut file: VecDeque<(Etat, i64)> = VecDeque::new(); file.push_back((initial.clone(), 0));
    while let Some((s, d)) = file.pop_front() {
        for (_, t) in successeurs_relaxes(&s, ops) {
            if vu.insert(t.clone()) {
                for p in t.difference(&s) { dist_atoms.entry(p.clone()).or_insert(d + 1); }
                file.push_back((t, d + 1));
            }
        }
    }
    dist_atoms
}

// h_max est None lorsque le but reste inatteignable dans la relaxation.
fn h_max(etat: &Etat, but: &Etat, ops: &[Operateur]) -> Option<i64> {
    let dr = distances_relaxees(etat, ops);
    let mut best: Option<i64> = None;
    for b in but.iter() {
        let v = *dr.get(b)?;
        best = Some(match best { None => v, Some(cur) => cur.max(v) });
    }
    best
}

// BFS reel pour calculer h*(A_t+1).
fn bfs(initial: &Etat, ops: &[Operateur]) -> (HashMap<Etat, i64>, Vec<Etat>) {
    let mut dist: HashMap<Etat, i64> = HashMap::new(); let mut ordre: Vec<Etat> = vec![initial.clone()]; let mut file: VecDeque<Etat> = VecDeque::new();
    dist.insert(initial.clone(), 0); file.push_back(initial.clone());
    while let Some(s) = file.pop_front() {
        let g = *dist.get(&s).unwrap_or(&0);
        for (_, t) in successeurs(&s, ops) {
            if !dist.contains_key(&t) { dist.insert(t.clone(), g + 1); ordre.push(t.clone()); file.push_back(t); }
        }
    }
    (dist, ordre)
}

// Grille et vocabulaire de base.
const COLS_GAUCHE: [i64; 3] = [0, 1, 2];
const COLS_DROITE: [i64; 2] = [4, 5];
const RANGS: [i64; 3] = [0, 1, 2];
fn prop(c: i64, r: i64) -> String { format!("at-x{c}-y{r}") }
fn singleton(s: &str) -> Etat { let mut e = BTreeSet::new(); e.insert(s.to_string()); e }
fn cases() -> Vec<(i64, i64)> {
    let mut v = Vec::new();
    for c in COLS_GAUCHE.iter().chain(COLS_DROITE.iter()) { for r in RANGS.iter() { v.push((*c, *r)); } }
    v.sort(); v.dedup(); v
}
fn initial() -> Etat {
    let mut s = BTreeSet::new();
    s.insert(prop(0, 0)); s.insert("plank-at-x1-y1".to_string());
    s.insert("key-at-x2-y0".to_string()); s.insert("treasure-at-x4-y1".to_string()); s
}
fn ops_de_base() -> Vec<Operateur> {
    let cs = cases(); let mut ops = Vec::new();
    for (c, r) in cs.iter() {
        for (dc, dr, direction) in [(1, 0, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud")] {
            let cible = (*c + dc, *r + dr);
            if cs.contains(&cible) {
                ops.push(Operateur { nom: format!("move-{direction}-x{c}-y{r}"), precond: singleton(&prop(*c, *r)), ajouts: singleton(&prop(cible.0, cible.1)), dels: singleton(&prop(*c, *r)) });
            }
        }
    }
    for (objet, c, r) in [("plank", 1, 1), ("key", 2, 0), ("treasure", 4, 1)] {
        let mut pre = BTreeSet::new(); pre.insert(prop(c, r)); pre.insert(format!("{objet}-at-x{c}-y{r}"));
        let mut aj = BTreeSet::new(); aj.insert(format!("holding-{objet}"));
        let mut dl = BTreeSet::new(); dl.insert(format!("{objet}-at-x{c}-y{r}"));
        ops.push(Operateur { nom: format!("pickup-{objet}"), precond: pre, ajouts: aj, dels: dl });
    }
    ops
}

// Le radeau etend A_t avec trois operateurs nommes.
fn ops_avec_radeau() -> Vec<Operateur> {
    let mut ops = ops_de_base();
    for r in RANGS.iter() {
        let mut pre = BTreeSet::new(); pre.insert(prop(2, *r)); pre.insert("holding-plank".to_string());
        let mut aj = BTreeSet::new(); aj.insert(prop(4, *r));
        let mut dl = BTreeSet::new(); dl.insert(prop(2, *r)); dl.insert("holding-plank".to_string());
        ops.push(Operateur { nom: format!("raft-across-y{r}"), precond: pre, ajouts: aj, dels: dl });
    }
    ops
}
fn buts() -> Vec<(&'static str, Etat)> {
    vec![("g1: balise (5,2)", singleton(&prop(5, 2))), ("g2: detenir le tresor", singleton("holding-treasure")), ("g3: detenir la cle", singleton("holding-key"))]
}
fn opt_txt(v: Option<i64>) -> String { match v { None => "inf".to_string(), Some(x) => x.to_string() } }

// Comparaison des bornes avant et apres l'extension du vocabulaire.
let etat_initial = initial();
let a_t = ops_de_base();
let a_t1 = ops_avec_radeau();
let liste_buts = buts();
let (dist_t1, s_atteint_1) = bfs(&etat_initial, &a_t1);
for (nom, but) in liste_buts.iter().take(2) {
    let hm_avant = h_max(&etat_initial, but, &a_t);
    let hm_apres = h_max(&etat_initial, but, &a_t1);
    let hstar1 = s_atteint_1.iter().filter_map(|s| if but.is_subset(s) { dist_t1.get(s) } else { None }).min().copied();
    let chaine = match (hm_apres, hstar1) { (Some(a), Some(h)) if a <= h => "OK", _ => "VIOLEE" };
    println!("{:28} h_max: A_t={} -> A_t+1={}   h*(A_t+1)={}   chaine h_max<=h*: {}", nom, opt_txt(hm_avant), opt_txt(hm_apres), opt_txt(hstar1), chaine);
};

g1: balise (5,2)             h_max: A_t=inf -> A_t+1=7   h*(A_t+1)=7   chaine h_max<=h*: OK


g2: detenir le tresor        h_max: A_t=inf -> A_t+1=6   h*(A_t+1)=6   chaine h_max<=h*: OK


## Résumé — le test opérationnel de la strate 7

| Pas | Mesure | Résultat |
|---|---|---|
| 1. Inatteignabilité vérifiée | énumération exhaustive | 36 états, $g_1, g_2$ absents — certificat exact |
| 2. Contrôle d'effort | BFS / A\*+$h_{max}$ / IDDFS(24) | les trois échouent, **même espace** : c'est un problème de vocabulaire |
| 3. Primitive nommée | `raft-across-y{r}` | coût : planchette consommée + détour $(1,1)$ |
| 4. Delta | buts, plans, espace | $\Delta = \{g_1, g_2\}$, plans exhibés ($h^*$ 7 et 6), l'espace atteignable 36 → 60 |

**Ce que le certificat couvre et ne couvre pas.** Le lake garantit les énoncés *structurels* : $step \subseteq stepR$ (`Strips.lean:58`), $run \subseteq runR$ (`Relaxation.lean:56`), $h^+ \le h^*$ (`Admissibility.lean:41`). Il ne dit **rien** du choix d'une primitive, ni de l'optimisation du chercheur : ce qui est certifié est la *garantie de l'heuristique*, ce qui est mesuré ici est *l'instance*. Les autres sémantiques (préférences, temporalité, incertitude) restent hors de ce certificat comme de cette mesure.

**Le protocole, réutilisable.** Toute revendication du type « l'agent a appris une nouvelle capacité » passe le même banc : (1) but inatteignable prouvé, (2) l'effort accru échoue, (3) la primitive est nommée et payante, (4) le delta est un ensemble ou un chiffre. Sans le pas 2, chercher davantage et élargir sont indiscernables ; sans le pas 3, le delta ne mesure rien.

## Limites et suites

Ce que ce banc ne fait **pas**, pour être lu honnêtement :

- **Ce n'est pas une étude de solveur**. Fast Downward et OR-Tools CP-SAT ([Planners-4](Planners-04-Fast-Downward-Python.ipynb), [Planners-7](../03-Advanced/Planners-07-OR-Tools-Python.ipynb)) traitent des espaces que ce mini-moteur ne peut pas énumérer ; ici, l'énumération exhaustive est précisément l'**instrument de mesure** — c'est elle qui transforme « on n'a pas trouvé » en certificat. Sur un espace exponentiel, le pas 1 devrait remplacer l'énumération par une preuve d'inatteignabilité (invariants, abstraction) — le protocole reste, l'instrument change.
- **$h^+$ exact n'est pas calculé** — NP-difficile en général ; la chaîne mesurée s'arrête à ses bornes calculables ($h_{max}$, $h^*$). Le companion formel [Planners-5b](Planners-05b-Lean-Relaxation-Lean.ipynb) exécute le lemme d'admissibilité dans le kernel Lean lui-même.
- **La synthèse reste hors Lean** : ce qui est certifié est la *garantie structurelle* de la relaxation, pas le choix de la primitive. Un `raft-across` mal fichu (qui traverserait sans planchette) passerait le même banc — l'exercice 2 montre le versant symétrique : une extension stérile y échoue.

**Suite naturelle** : le différentiel se généralise en remove-differential (quelle classe de plans **disparaît** quand on retire une primitive) et en coût-differential (même vocabulaire, coût des actions changé) — le même protocole à quatre pas s'applique, seul le pas 3 change de geste.

## Exercices

### Exercice 1 — Une primitive *gratuite* : le deltaplane

Ajoutez l'opérateur `deltaplane-y{r}` : précondition être en $(0,r)$ (bord **gauche**), effet atterrir en $(4,r)$, **sans planchette**. Recalculez $\Delta$ et $h^*(g_1)$ avec ce vocabulaire $A_{t+1}'$. Le delta change-t-il ? Le coût, oui ou non, et pourquoi le pas 3 du protocole exige-t-il de le dire ?

In [10]:
// Exercice 1 a completer : construire A_t1_prime avec deltaplane-y{r}, recalculer
// le delta de buts et h*(g1), puis comparer au radeau.
// A_t1_prime = ...
// result_exo1 = None  # TODO etudiant : (delta, hstar_g1)
// TODO etudiant
let _result_exo1: Option<(Vec<&'static str>, i64)> = None;  // TODO etudiant
println!("Exercice 1 a completer : delta et h*(g1) avec le deltaplane.");

Exercice 1 a completer : delta et h*(g1) avec le deltaplane.


### Exercice 2 — Une extension *stérile* : jeter la planchette

Ajoutez l'opérateur `discard-plank` : précondition `holding-plank`, effet suppression de `holding-plank` (rien d'ajouté). Le vocabulaire s'élargit pourtant — mesurez $\Delta$ et la taille de l'ensemble atteignable. Que démontre ce contre-exemple sur le protocole ?

In [11]:
// Exercice 2 a completer : construire A_t1_sterile = A_t + discard-plank,
// verifier que Delta = ensemble vide et que l'espace atteignable ne change pas.
// result_exo2 = None  # TODO etudiant : (delta, taille_atteignable)
// TODO etudiant
let _result_exo2: Option<(Vec<&'static str>, usize)> = None;  // TODO etudiant
println!("Exercice 2 a completer : delta et taille de l'espace avec discard-plank.");

Exercice 2 a completer : delta et taille de l'espace avec discard-plank.


### Exercice 3 — La chaîne $h_{max} \le h^+ \le h^*$ sur $g_3$

Sur le but $g_3$ (détenir la clé), calculez $h_{max}$ et $h^*$ dans $A_t$. Sont-ils égaux ? Trouvez l'argument (en une phrase) qui explique pourquoi la relaxation ne gagne **rien** sur ce but, alors qu'elle gagnait 5 unités sur le but conjonctif de la rive gauche ($h_{max} = 3$ contre $h^* = 8$).

In [12]:
// Exercice 3 a completer : h_max(g3) et h*(g3) dans A_t, verdict egalite/strict,
// et la phrase d'explication dans explication_exo3.
// result_exo3 = None   # TODO etudiant : (hmax_g3, hstar_g3)
// explication_exo3 = ""  # TODO etudiant
// TODO etudiant
// TODO etudiant
let _result_exo3: Option<(i64, i64)> = None;   // TODO etudiant
let _explication_exo3: &str = "";  // TODO etudiant
println!("Exercice 3 a completer : h_max(g3), h*(g3), explication.");

Exercice 3 a completer : h_max(g3), h*(g3), explication.


## À retenir

**Navigation** : [Index](../../README.md) | [Jumeau Python](Planners-05c-Differentiel-Atteignabilite-Python.ipynb)

| Leçon | Mesure dans ce carnet |
|---|---|
| L'inatteignabilité se démontre, elle ne se suppose pas | BFS exhaustif dans `A_t` : `36 etats, 36 expansions` — `g1 ... atteignable=False h*=infini`, `g2 ... atteignable=False`, `g3: detenir la cle atteignable=True h*=3` |
| Un échec de recherche n'est pas un certificat : trois budgets, même verdict | `g1` et `g2` échouent aux trois niveaux d'effort : `BFS ECHEC( 36)`, `A*+h_max ECHEC( 36)`, `IDDFS(24) ECHEC( 4151)` ; `g3` passe aux trois (`OK( 36)`, `OK( 4)`, `OK( 24)`) |
| La relaxation ne peut pas inventer la rive droite | `g1: balise (5,2): h_max dans A_t = INFINI (relaxement inatteignable)` — et le carnet cite sa garantie : `le lake garantit h+ <= h* en Admissibility.lean:41` |
| La chaîne d'admissibilité se vérifie sur le but conjonctif | `h* (reel, BFS uniforme) = 8`, `h_max (relaxe, exact) = 3`, `chaine verifiee : h_max (3) <= h* (8) -> STRICT` |
| Le différentiel se mesure : une primitive nommée, trois opérateurs | `A_t : 41 operateurs` ; `A_t+1 : 44 operateurs (primitive ajoutee : ['raft-across-y0', 'raft-across-y1', 'raft-across-y2'])` |
| Ce que la primitive ouvre, en états et en coûts | `|A_t| = 36 etats -> |A_t+1| = 60 etats (+24)` ; `g1 ECHEC -> OK h*(A_t+1) = 7 NOUVELLEMENT ATTEIGNABLE`, `g2 ECHEC -> OK h* = 6`, `g3 OK -> OK h* = 3` |

## Vérifiez votre compréhension

**Question 1 — Pourquoi le tableau du pas 2 (trois algorithmes) n'ajoute-t-il rien au verdict du pas 1 (BFS exhaustif), et pourquoi le faire quand même ?**

Le pas 1 rend `Ensemble atteignable dans A_t : 36 etats, 36 expansions (BFS exhaustif)` ; le pas 2 rend `Verdict du controle : g1 et g2 echouent aux TROIS niveaux d'effort. L'espace atteignable est le MEME (36 etats) quel que soit le budget.`

<details><summary>Réponse</summary>

Parce que le pas 1 **démontre** (énumération exhaustive des 36 états : rive droite absente), tandis que le pas 2 teste la **robustesse du verdict** face à l'objection « tu n'as pas cherché assez ». IDDFS pousse le budget à `4151` expansions — deux ordres de grandeur au-delà des `36` de l'exhaustif — et ne trouve rien de plus : c'est la signature d'une barrière structurelle (le gouffre sans radeau), pas d'un budget trop court. La mesure asymétrique est la leçon : `g3 OK(4)` en A* contre `OK(36)` en BFS montre aussi ce que le guidage achète quand le but **est** atteignable.

</details>

**Question 2 — Après l'ajout du radeau, h_max(g1) passe de l'infini à 7 : que vérifie la ligne `chaine h_max<=h*: OK` et d'où vient la garantie ?**

Le pas 4 rend `g1: balise (5,2) h_max: A_t=inf -> A_t+1=7 h*(A_t+1)=7 chaine h_max<=h*: OK` et le carnet cite `le lake garantit h+ <= h* en Admissibility.lean:41`.

<details><summary>Réponse</summary>

Elle vérifie numériquement ce que le lake prouve formellement : `h_max <= h+ <= h*` (h_max majore par le fait le plus cher du but relaxé ; h+ est le coût du plan relaxé ; l'admissibilité de la relaxation est le théorème d'`Admissibility.lean:41`, prouvé sans `sorry` dans le companion `Planners-05b-Lean-Relaxation-Lean`). Sur g1 après ajout : `h_max = 7 = h*` — égalité, jamais inversion. Et sur le but conjonctif du contrôle, l'écart était strict (`3 < 8`) : la chaîne autorise l'égalité, elle interdit la surestimation. C'est ce qui rend h_max utilisable pour **certifier** l'inatteignabilité : `h_max = INFINI` dans A_t ne peut pas venir d'une sous-estimation.

</details>

## Pour aller plus loin

- [Index de la sous-série 02-Classical](README.md) — le parcours principal et les paliers d'approfondissement (ce carnet est le palier 5c).
- [Index de la série Planners](../README.md) — la place du protocole de mesure dans la série.

Les carnets voisins sont cités en prose, sans lien. `Planners-05b-Lean-Relaxation-Lean` prouve le théorème (`h+ <= h*`) que ce carnet applique et cite à sa source ; `Planners-05-Heuristics-Python` construit les heuristiques (h_max, h+, h_landmark) dont la chaîne est mesurée ici ; `Planners-06-Domains-Python` montre les domaines classiques où ces questions d'atteignabilité se posent à grande échelle. La suite du travail est déjà posée dans le carnet : l'exercice 1 mesure le delta d'une primitive **gratuite** (le deltaplane), l'exercice 2 d'une extension **stérile** (jeter la planchette), l'exercice 3 la chaîne `h_max <= h+ <= h*` sur g3.